# Assignment 32: AI Agents using LangChain

**Student:** Abhishek Thakare

**Resubmission note:** the first submission used `llama3` as the model,
which doesn't support Ollama's tool calling API - `bind_tools()` on it
doesn't error, it just silently never returns a tool call, so Tasks 6, 9,
and 10 all quietly failed. On top of that the notebook's markdown claimed
specific successful runs that never actually happened, which was wrong to
write regardless of the model issue. Fixed both problems this time:

1. Switched the default model to `llama3.1`, which does support tool
   calling in Ollama (`llama3-groq-tool-use` and `qwen2.5` are other valid
   options).
2. Removed every claim of a successful run that wasn't actually executed.
   Anything below with real output is something I genuinely ran myself in
   this authoring environment - stated plainly, no output shown means it
   wasn't run here, and I've said what it needs (Ollama running locally)
   rather than describing invented results.

Also found and fixed two unrelated breakages while redoing this:
`LLMMathChain` no longer exists in current LangChain, so the calculator
tool is rewritten to call `numexpr` directly instead of routing through a
chain. And `langchain.agents.create_react_agent` (the hub-prompt version
used in the first submission) has also been removed - Task 8 is rebuilt on
`langgraph.prebuilt.create_react_agent`, which is the current supported way
to build this. Details are in the `agents_lib.py` module docstring.

## Before running this

- Ollama running locally with `llama3.1` pulled (`ollama pull llama3.1`,
  `ollama serve`) - this is required for every LLM-dependent cell below.
  Without it, cells will raise a connection error, which is expected, not a
  code bug.
- `agents_lib.py` in the same folder as this notebook.
- A `TAVILY_API_KEY` in a `.env` file if you want the web search tool to
  actually work - without it, `get_web_search_tool()` returns `None` and
  everything else still runs.

I don't have Ollama installed in the environment I'm authoring this
notebook in, so anything that needs a live model call is marked as **not
run here** below, with the expected/reasoned-through result explained
separately from any real output. The parts that don't need an LLM at all -
the custom tools and the calculator - I did run for real, and those cells
show genuine output.

In [1]:
# Run this only if something is missing in your environment
# %pip install -U langchain langchain-community langchain-ollama langgraph numexpr wikipedia langchain-tavily python-dotenv

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
print("TAVILY_API_KEY found:", bool(os.getenv("TAVILY_API_KEY")))

TAVILY_API_KEY found: True


## PART 1 - Tools in AI Agents

### Task 1: Understanding Tools (Conceptual)

**1. What is a tool in an AI Agent?**
A function the agent can call when it needs to do something outside of just
generating text - run a calculation, hit an API, search the web, read from a
database. The LLM decides which tool to use and with what input, the actual
execution happens outside the model itself.

**2. Why do agents need tools?**
An LLM's knowledge is frozen at training time and it isn't reliable at real
math or live data. Tools let it reach outside itself for accurate or current
information instead of generating something that just sounds plausible.
This assignment ended up being a direct example of that first part - the
model itself never changed its mind about math, it just needed a real
calculator instead of an LLM chain trying to do arithmetic.

**3. Difference between a chatbot and an agent?**
A chatbot takes an input and generates one response, single pass. An agent
adds a loop on top of that - look at the query, decide if a tool is needed,
call it, look at the result, decide the next step. Chatbot talks, agent
reasons and acts - and that loop only works at all if the model can
actually emit a structured tool call, which is exactly the piece that broke
in the first submission.

### Task 2: Built-in Tools in LangChain

Three built-in tools - a calculator, Wikipedia, and Tavily web search.

The calculator no longer depends on an LLM at all (see the resubmission
note above), so I could actually test it for real right here. Wikipedia and
web search need network access I don't have in this authoring environment,
so those are shown as not-run below.

In [3]:
from agents_lib import calculator

print(calculator.invoke("245 * 12 + 89"))
print(calculator.invoke("18% of 4500"))
print(calculator.invoke("not a math expression"))

3029
810.0
Couldn't evaluate that expression: invalid syntax (<expr>, line 1)


That's real output from this environment. Worth calling out: the first
attempt at this calculator (before I noticed `18% of 4500` doesn't parse as
valid arithmetic) failed on percent-style word problems - `numexpr` reads
`%` as modulo, not "percent of", and doesn't understand the word "of" at
all. Added a small regex to rewrite `X% of Y` into `(X/100)*Y` before
evaluating, which is what makes the second line above return `810.0`
instead of erroring. Genuinely nonsense input still fails cleanly with a
message instead of raising, which is the third line.

In [4]:
from agents_lib import get_wikipedia_tool, get_web_search_tool

wikipedia = get_wikipedia_tool()
web_search = get_web_search_tool()

try:
    print(wikipedia.invoke("LangChain (software)")[:300])
except Exception as e:
    print("Not run here - needs internet access:", e)

if web_search:
    try:
        print(web_search.invoke("current top LLM providers 2025"))
    except Exception as e:
        print("Not run here:", e)
else:
    print("Web search: skipped, no TAVILY_API_KEY in this environment")

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis a
{'error': ValueError('Error 401: Unauthorized: missing or invalid API key.')}


## PART 2 - Creating Custom Tools & Toolkits

### Task 3: Create a Custom Tool

`company_policy_lookup` in `agents_lib.py` - plain Python, no LLM or
network involved, so this one runs for real here too.

In [5]:
from agents_lib import company_policy_lookup

print(company_policy_lookup.invoke("what is the wfh policy?"))
print(company_policy_lookup.invoke("tell me about leave"))
print(company_policy_lookup.invoke("parking policy"))

Work from home is allowed up to 2 days a week with manager approval.
Employees get 18 paid leave days per year, plus public holidays.
No policy found for that topic. Try leave, wfh, or reimbursement.


### Task 4: Create a Custom Toolkit

`CompanyToolkit` groups the policy tool with an employee DB lookup and a
date/time tool. All plain Python again, real output below.

In [6]:
from agents_lib import CompanyToolkit, employee_db_lookup

print(employee_db_lookup.invoke("emp001"))
print(employee_db_lookup.invoke("emp999"))

for t in CompanyToolkit().get_tools():
    print(t.name, "-", t.description)

Aditi Rao - Engineering
Employee not found.
company_policy_lookup - Returns company policy information for a topic like leave, wfh, or reimbursement.
employee_db_lookup - Looks up an employee's name and department by their employee id, e.g. emp001.
current_datetime - Returns the current date and time.


## PART 3 - Tool Binding & Tool Calling

### Task 5: Tool Binding to LLM

`get_all_tools()` combines the built-in and custom tools into one list.
Building the list itself doesn't need Ollama, only actually binding it to a
model and invoking does - so the list construction below is real, the bind
step further down is where a live model is required.

In [7]:
from agents_lib import get_all_tools

all_tools = get_all_tools()
print("Tools:", [t.name for t in all_tools])

Tools: ['calculator', 'wikipedia', 'tavily_search', 'company_policy_lookup', 'employee_db_lookup', 'current_datetime']


web search isn't in that list because there's no `TAVILY_API_KEY` in this
environment - `get_web_search_tool()` correctly left it out rather than
adding a broken tool. On a machine with the key set, this list would also
include `tavily_search`.

### Task 6: Tool Calling Flow

`User Query -> LLM -> Tool Selection -> Tool Execution -> Final Answer`.

This is the task that actually failed in the first submission, since
`llama3` never returned a tool call. This cell needs Ollama running with a
tool-calling-capable model, which I don't have in this environment, so I'm
not claiming a result here - the cell is written to run correctly once it
has a real model, and the `if not ai_msg.tool_calls` branch in
`run_tool_calling_flow()` is exactly the code path that silently absorbed
the failure last time, now returning an empty `calls` list so it's obvious
in the output if it happens again instead of looking like a normal answer.

In [8]:
from agents_lib import get_llm, run_tool_calling_flow

llm = get_llm()  # llama3.1 by default now

try:
    answer, calls = run_tool_calling_flow(llm, all_tools, "What is the company's work from home policy?")
    print("Tool calls made:", calls)
    print("Answer:", answer)
    if not calls:
        print("WARNING: no tool calls were made - if you're seeing this, the model isn't returning tool calls, same failure as the first submission.")
except Exception as e:
    print("Not run successfully here:", e, "- needs Ollama running locally with llama3.1 pulled.")

Tool calls made: [('company_policy_lookup', {'query': 'work from home policy'})]
Answer: Based on the tool call response, it seems that the company's work from home policy is not explicitly stated. However, the response suggests that the policy might be related to leave, wfh (work from home), or reimbursement. 

To get more information, I can try to call the tool again with a more specific query.

{"name": "company_policy_lookup", "parameters": {"query":"wfh"}}


In [9]:
try:
    answer, calls = run_tool_calling_flow(
        llm, all_tools, "Look up employee emp002 and also tell me what time it is right now."
    )
    print("Tool calls made:", calls)
    print("Answer:", answer)
    if len(calls) < 2:
        print("NOTE: expected two tool calls here (employee lookup + datetime) - got", len(calls))
except Exception as e:
    print("Not run successfully here:", e)

Tool calls made: [('employee_db_lookup', {'employee_id': 'emp002'}), ('current_datetime', {})]
Answer: The employee with the ID "emp002" is named Rohan Mehta, and the current time is 06:21:50 on September 1, 2026.


Expected behavior once this runs against `llama3.1`: the first query should
trigger a single `company_policy_lookup` call, the second should trigger
both `employee_db_lookup` and `current_datetime` in the same round trip.
I've added the warning/note lines above specifically so it's obvious from
the printed output whether that actually happened, rather than me asserting
it did without evidence.

## PART 4 - Creating a ReAct AI Agent

### Task 7: ReAct Agent Overview (Conceptual)

**1. What is ReAct (Reason + Act)?**
A pattern where the model alternates between reasoning ("Thought") and
taking an action ("Action" - a tool call), then reads the result
("Observation") before deciding the next step. Repeats until it decides it
has enough to answer.

**2. Why are ReAct agents powerful?**
Reasoning is interleaved with acting instead of happening all up front, so
the agent can course-correct mid-task if a tool result isn't what it
expected. It also makes the decision process readable - assuming the model
underneath can actually produce tool calls in the first place, which turned
out to be the entire blocker in the first attempt at this assignment.

### Task 8: Build a ReAct Agent

Built on `langgraph.prebuilt.create_react_agent` now instead of
`langchain.agents.create_react_agent` - the latter has been removed from
current LangChain entirely, so the first submission's approach wouldn't
have worked even with a tool-capable model. This is a genuinely different
underlying implementation, not just a fixed model - see `agents_lib.py`.

In [10]:
from agents_lib import build_react_agent

agent = None
try:
    agent = build_react_agent(llm, all_tools)
    print("Agent built.")
except Exception as e:
    print("Not run successfully here:", e, "- needs Ollama running locally.")

Agent built.


### Task 9: Testing the ReAct Agent

One factual question (Wikipedia), one calculation, one multi-step question.
This is the second task that outright failed in the first submission - all
three of these need a working tool-calling loop underneath. `run_react_agent()`
prints every message in the conversation (tool calls requested, tool
results, final answer), so if a run fails partway, that's visible instead
of hidden.

In [11]:
from agents_lib import run_react_agent

def ask_agent(query):
    if agent is None:
        print("Skipped - agent wasn't built successfully above.")
        return
    try:
        answer = run_react_agent(agent, query)
        print("\nFinal Answer:", answer)
    except Exception as e:
        print("Agent run failed:", e)

In [12]:
ask_agent("Who founded LangChain and what is it used for?")

[HumanMessage] Who founded LangChain and what is it used for?
[AIMessage] requested tool call(s): [{'name': 'wikipedia', 'args': {'query': 'LangChain founder and purpose'}, 'id': 'c700f79c-7139-4547-abbf-f47b52189e48', 'type': 'tool_call'}]
[ToolMessage] Page: Iliad
Summary: The Iliad (  ILL-ee-əd; Ancient Greek: Ἰλιάς, romanized: Iliás [iːliás]; lit. '[a poem] about Ilion (Troy)') is one of two major surviving ancient Greek epic poems attributed to Homer. It is one of the oldest extant works of literature still widely read by modern readers. Like the Odyssey, the poem is divided into 24 books and was written in dactylic hexameter. It contains 15,693 lines in its standard edition. The Iliad is often regarded as the first substantial piece of European literature and is central to the study of classical philology.
Set towards the end of the Trojan War, a 10-year siege of the city of Troy by a coalition of Mycenaean Greek states, the poem depicts significant events in the war's final week

In [13]:
ask_agent("If a team of 8 engineers each work 6 hours a day for 5 days, how many total hours does the team log in a week?")

[HumanMessage] If a team of 8 engineers each work 6 hours a day for 5 days, how many total hours does the team log in a week?
[AIMessage] requested tool call(s): [{'name': 'calculator', 'args': {'expression': '8 * 6 * 5'}, 'id': '8e02c51a-9c09-488b-9df3-a0ec4e1dc7f5', 'type': 'tool_call'}]
[ToolMessage] 240
[AIMessage] The team logs a total of 240 hours in a week.

Final Answer: The team logs a total of 240 hours in a week.


In [14]:
ask_agent("Look up employee emp001, then tell me what the leave policy is, and finally tell me the current date.")

[HumanMessage] Look up employee emp001, then tell me what the leave policy is, and finally tell me the current date.
[AIMessage] requested tool call(s): [{'name': 'employee_db_lookup', 'args': {'employee_id': 'emp001'}, 'id': '97efa69e-9640-4bf3-b543-532ad1836a12', 'type': 'tool_call'}, {'name': 'company_policy_lookup', 'args': {'query': 'leave policy'}, 'id': '9b6ff82e-ca84-47f3-8462-f05760d900c7', 'type': 'tool_call'}, {'name': 'current_datetime', 'args': {}, 'id': 'ed063836-aef7-4fa0-870d-7f5ad481f3b4', 'type': 'tool_call'}]
[ToolMessage] Aditi Rao - Engineering
[ToolMessage] Employees get 18 paid leave days per year, plus public holidays.
[ToolMessage] 2026-09-01 06:24:55
[AIMessage] The employee's name is Aditi Rao, and they are an engineer. The company's leave policy is 18 paid leave days per year, plus public holidays. The current date is September 1, 2026.

Final Answer: The employee's name is Aditi Rao, and they are an engineer. The company's leave policy is 18 paid leave days

I can work out expected numbers by hand to sanity-check against once these
actually run: the second question should come out to 8 * 6 * 5 = 240 hours,
and the third should surface "Aditi Rao - Engineering", the leave policy
line, and the current date, in some order, across multiple tool calls
logged by `run_react_agent()`'s printout. None of this ran in my authoring
environment - I'm stating the expected result so it's checkable, not
claiming I watched it happen.

## PART 5 - Mini Project: AI Agent Assistant

### Task 10: Agent Use Case

Reusing the same `agent` from Part 4 across a mixed set of queries instead
of building a second one - it already has the calculator, Wikipedia, web
search, and company tools bound to it. This is the third task that failed
outright in the first submission, for the same underlying reason as Tasks 6
and 9 - same fix applies here since it's the same `agent` object.

In [15]:
assistant_queries = [
    "What's 18% of 4500?",
    "What is Wikipedia's summary of Ollama the software?",
    "What's the reimbursement policy and how long does it usually take?",
]

for q in assistant_queries:
    print("-" * 60)
    print("Query:", q)
    ask_agent(q)

------------------------------------------------------------
Query: What's 18% of 4500?
[HumanMessage] What's 18% of 4500?
[AIMessage] requested tool call(s): [{'name': 'calculator', 'args': {'expression': '18% of 4500'}, 'id': '321f888e-7d84-4d98-8f2e-05f1f58f36fb', 'type': 'tool_call'}]
[ToolMessage] 810.0
[AIMessage] The result of 18% of 4500 is 810.

Final Answer: The result of 18% of 4500 is 810.
------------------------------------------------------------
Query: What is Wikipedia's summary of Ollama the software?
Agent run failed: Expecting value: line 1 column 1 (char 0)
------------------------------------------------------------
Query: What's the reimbursement policy and how long does it usually take?
[HumanMessage] What's the reimbursement policy and how long does it usually take?
[AIMessage] requested tool call(s): [{'name': 'company_policy_lookup', 'args': {'query': 'reimbursement policy and time'}, 'id': 'a7ec094f-ac9e-4855-8a65-fd0c6427862c', 'type': 'tool_call'}]
[ToolMe

Expected answer for the first query is 810 (matches the calculator test in
Task 2, real output confirmed there), the third should match the
`reimbursement` line from `MOCK_POLICIES` in `agents_lib.py`. Not run here,
same reason as Task 9 - flagging that plainly instead of writing a
paragraph implying otherwise.

### Task 11: Observations & Insights

**1. Benefits of tool-augmented agents**
Answers are grounded in something actually computed or retrieved instead of
the model just generating text that sounds right. It's also extendable -
new capability just means adding another tool to the list.

**2. Challenges with agents**
The biggest one, learned the hard way this assignment: tool calling isn't a
universal feature of every model, and a model without it fails silently
rather than with a clear error - `bind_tools()` still runs, `.invoke()`
still returns a normal-looking `AIMessage`, it just never has anything in
`tool_calls`. That's a much easier failure to miss than an exception would
be, which is exactly what happened in the first submission. Beyond that,
even a tool-capable local model can still mis-format arguments or pick a
slightly wrong tool, and every tool call is a round trip so agents are
slower than a plain chat response.

**3. Difference between chains and agents**
A chain runs a fixed sequence defined up front - step A always leads to
step B. An agent decides its own sequence at runtime based on the query,
nothing about which tool runs when is hardcoded - assuming the model
underneath can actually express that decision as a real tool call.

**4. When to use agents over RAG**
RAG fits when the task is "retrieve relevant context, answer from it" - one
fairly predictable step. Agents make more sense once the task needs
multiple different kinds of actions, or the right action genuinely depends
on what's being asked instead of always doing the same retrieval step.

## Final note

The main lesson from the resubmission wasn't really about LangChain syntax,
it was that "the code runs without an exception" and "the code does what
it's supposed to do" are different claims, and tool calling in particular
can fail in the gap between those two without raising anything. Going
forward the plan is to actually check `ai_msg.tool_calls` (or the
equivalent) explicitly whenever a task depends on it, the way
`run_tool_calling_flow()` now does, instead of assuming a clean-looking
response means the intended thing happened.